# Unit 3 — Multimedia and Interactive Visualization (UE24CS342AA9)
## Activity Notebook: The Fraud-Ops War Room

### Scenario

You've joined the **Fraud Operations** team at **PayKart**, a card-payments company. Last month's numbers looked wrong and
the Head of Fraud Ops has called a review. You have 30 days of transactions with the automated detector's score and the
investigators' confirmed labels (`unit3_ppt7_card_transactions.csv`). Their requests:

1. *"I don't want a list of alerts. Show me the whole month first — is anything unusual going on?"*
2. *"The investigators keep saying 'night-time, far-from-home' transactions are risky. Show me, across ALL our views at
   once, what those transactions look like."*
3. *"Engineering wants to build the interactive filter tool on our full history: 50 million transactions. Will a slider
   still feel instant?"*
4. *"The detector threshold is 0.5 and the team is drowning in alerts. We can review at most **150 alerts a month**. Where
   should the threshold be?"*
5. *"Give the investigators a tool where they can poke at the flagged cases themselves."*
6. *"Someone says it was a fraud ring in one city. Prove it — or kill the idea."*

Cells marked **`# TODO`** are for you to complete. Markdown cells marked **Reflection** are for you to answer in your own
words. Interactive (plotly) figures support zoom, pan, hover, selection and clicking.

## Part 0 — Setup (given)

Run this cell as-is. It loads `tx` and adds `night` (hour < 6) and a readable `label` column.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

np.random.seed(0)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
# If plotly figures come out blank: classic Jupyter -> pio.renderers.default = "notebook";
# Google Colab -> "colab"; VS Code / JupyterLab work out of the box.
import plotly.io as pio

tx = pd.read_csv("unit3_ppt7_card_transactions.csv", parse_dates=["timestamp"])
tx["night"] = tx.hour < 6
tx["label"] = np.where(tx.is_fraud == 1, "confirmed fraud", "legitimate")

print(f"{len(tx):,} transactions | {tx.is_fraud.sum()} confirmed fraud | days {tx.day.min()}-{tx.day.max()}")
tx.head()

## Task 1 — Overview First (Request #1)

**Real-world usage:** Fraud dashboards start from a whole-period overview before any alert list.

**Why this technique:** Shneiderman's mantra — **overview first**, then zoom and filter, then details (Slide 30); a
**reordered** matrix makes patterns pop out (Slide 39).

**TODO:**
1. Plot daily transaction counts with daily confirmed fraud on a second axis.
2. Build a city × day matrix of confirmed-fraud **counts**, reorder rows and columns by their totals, and show it as a
   heatmap with labelled axes.
3. Print the (city, day) cells with the most fraud.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 7))
# TODO 1
per_day = tx.groupby("day").size()
fraud_per_day = tx[tx.is_fraud == 1].groupby("day").size().reindex(per_day.index, fill_value=0)
axes[0].bar(per_day.index, per_day.values, color="lightsteelblue", label="all transactions")
axes[0].set(xlabel="Day", ylabel="Transactions", title="Daily volume and confirmed fraud")
ax_f = axes[0].twinx()
ax_f.spines["right"].set_visible(True)
ax_f.plot(fraud_per_day.index, fraud_per_day.values, "o-", color="crimson", label="confirmed fraud")
ax_f.set_ylabel("Confirmed fraud", color="crimson")

# TODO 2
fraud_only = tx[tx.is_fraud == 1]
heat = (fraud_only.groupby(["city", "day"]).size().unstack(fill_value=0)
        .reindex(index=sorted(tx.city.unique()), columns=sorted(tx.day.unique()), fill_value=0))
heat = heat.loc[heat.sum(axis=1).sort_values(ascending=False).index,
                heat.sum(axis=0).sort_values(ascending=False).index]
im = axes[1].imshow(heat.values, aspect="auto", cmap="Reds")
axes[1].set_xticks(range(heat.shape[1])); axes[1].set_xticklabels(heat.columns, fontsize=7)
axes[1].set_yticks(range(heat.shape[0])); axes[1].set_yticklabels(heat.index)
axes[1].set(xlabel="Day (reordered by total fraud)", ylabel="City (reordered by total fraud)",
            title="Confirmed fraud count: city x day")
fig.colorbar(im, ax=axes[1], label="fraud count")

plt.tight_layout(); plt.show()

# TODO 3
print("Top (city, day) cells by confirmed fraud:")
print(heat.stack().sort_values(ascending=False).head(8))

**Reflection:** What did the overview show that an alert list sorted by detector score would *not* have shown? Name the
lecture concept (Slide 38 or 39) that describes how you found it.

_Your answer:_

## Task 2 — Linked Views for "Night-Time, Far-From-Home" (Request #2)

**Real-world usage:** Investigators validate rules of thumb by brushing them across a coordinated dashboard.

**Why this technique:** **Cross-view brushing / linked highlighting** reveals how a selection in one view is distributed
in all the others (Slides 8–12).

**TODO:**
1. Define the brush: `night == True` **and** `distance_from_home_km > 100`.
2. Draw four coordinated views with the brushed items in red over all items in grey: (a) hour vs. distance scatter
   (log y), (b) merchant category counts, (c) city counts, (d) amount histogram (log10).
3. Print: number brushed, fraud rate inside the brush vs. overall, and the share of *all* fraud the brush captures.

In [ ]:
# the dataset's column names for merchant category / customer may differ slightly, so look them up
cat_col = next(c for c in tx.columns if "categ" in c.lower())
print("Using category column:", cat_col)

# TODO 1
brush = tx.night & (tx.distance_from_home_km > 100)

fig, axes = plt.subplots(1, 4, figsize=(19, 4))
# TODO 2
sel = tx[brush]

# (a) hour vs distance
axes[0].scatter(tx.hour, tx.distance_from_home_km, s=4, color="lightgrey")
axes[0].scatter(sel.hour, sel.distance_from_home_km, s=6, color="red")
axes[0].set(yscale="log", xlabel="Hour", ylabel="Distance from home (km)", title="Hour vs distance")

# (b) merchant category counts
cats = tx[cat_col].value_counts()
axes[1].bar(cats.index, cats.values, color="lightgrey")
axes[1].bar(cats.index, sel[cat_col].value_counts().reindex(cats.index, fill_value=0).values, color="red")
axes[1].set(title="Merchant category", ylabel="Count"); axes[1].tick_params(axis="x", rotation=60)

# (c) city counts
cities = tx.city.value_counts()
axes[2].bar(cities.index, cities.values, color="lightgrey")
axes[2].bar(cities.index, sel.city.value_counts().reindex(cities.index, fill_value=0).values, color="red")
axes[2].set(title="City", ylabel="Count"); axes[2].tick_params(axis="x", rotation=60)

# (d) amount histogram (log10)
bins = np.linspace(np.log10(tx.amount_inr.min() + 1), np.log10(tx.amount_inr.max() + 1), 40)
axes[3].hist(np.log10(tx.amount_inr + 1), bins=bins, color="lightgrey")
axes[3].hist(np.log10(sel.amount_inr + 1), bins=bins, color="red")
axes[3].set(xlabel="log10(amount in INR)", ylabel="Count", title="Amount")

plt.tight_layout(); plt.show()

# TODO 3
overall_rate = tx.is_fraud.mean()
brush_rate = sel.is_fraud.mean()
captured = sel.is_fraud.sum() / tx.is_fraud.sum()
print(f"Brushed: {len(sel):,} transactions")
print(f"Fraud rate inside brush: {brush_rate:.1%} vs overall {overall_rate:.1%}")
print(f"Share of all fraud captured by the brush: {captured:.1%}")

**Reflection:** The brush has a very high fraud *rate* — but what share of all fraud does it *capture*? What does that
tell the investigators about relying on this rule alone, and which linked view hinted at where the rest of the fraud is?

_Your answer:_

## Task 3 — Will the Slider Still Feel Instant? (Request #3)

**Real-world usage:** Interactive tools are sized against latency budgets before they're built.

**Why this technique:** **Dynamic queries** need feedback within ~0.1 s (Slides 15–16).

**TODO:**
1. Build a 1-million-row table by repeating `tx`, and measure the time of the filter
   `(detector_score >= 0.6) & (amount_inr > 3000)` (take the best of 5 runs).
2. Compute the per-item cost (µs) and the maximum number of rows filterable within 0.1 s on this machine.
3. Estimate the response time for 50,000,000 rows and the % of rows that would need to be skipped (pre-indexed /
   pre-aggregated) to meet the 0.1 s budget.

In [ ]:
big = pd.concat([tx] * 182, ignore_index=True)

# TODO 1
def run_filter():
    return big[(big.detector_score >= 0.6) & (big.amount_inr > 3000)]

timings = []
for _ in range(5):
    t0 = time.perf_counter()
    run_filter()
    timings.append(time.perf_counter() - t0)
best_s = min(timings)

# TODO 2
per_item_us = best_s / len(big) * 1e6
max_rows = 0.1 / (per_item_us * 1e-6)

# TODO 3
t_50m = per_item_us * 1e-6 * 50_000_000
skip_pct = max(0.0, (1 - 0.1 / t_50m) * 100)

print(f"{len(big):,} rows in {best_s*1000:.1f} ms -> {per_item_us:.4f} us/row -> max {max_rows:,.0f} rows in 0.1 s")
print(f"50M rows -> {t_50m:.2f} s; must skip {skip_pct:.1f}% of rows to stay under 0.1 s")

**Reflection:** Besides indexing, name one **reduce** technique from the lecture (Slide 5 / 14) that would let the slider
stay instant on 50M rows, and what the analyst would lose by using it.

_Your answer:_

## Task 4 — Steer the Threshold to the Team's Capacity (Request #4)

**Real-world usage:** Alert thresholds are set against investigator capacity, not just statistical optimality.

**Why this technique:** **Human-in-the-loop** steering of a model parameter (Slide 37) evaluated with **precision, recall,
F1** (Slides 40–41).

**TODO:**
1. For thresholds 0.05, 0.10, …, 0.95 compute alerts (flagged count), TP, FP, FN, precision, recall and F1.
2. Find the threshold with the **highest recall** among those producing **≤ 150 alerts**. Also find the best-F1 threshold.
3. Plot the precision–recall curve and mark the current threshold (0.5), the capacity-constrained choice and the best-F1
   choice.
4. Print a one-line recommendation.

In [ ]:
rows = []
# TODO 1
for thr in np.round(np.arange(0.05, 0.96, 0.05), 2):
    flagged_mask = tx.detector_score >= thr
    tp = int((flagged_mask & (tx.is_fraud == 1)).sum())
    fp = int((flagged_mask & (tx.is_fraud == 0)).sum())
    fn = int((~flagged_mask & (tx.is_fraud == 1)).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    rows.append({"threshold": thr, "alerts": tp + fp, "tp": tp, "fp": fp, "fn": fn,
                 "precision": precision, "recall": recall, "f1": f1})

pr = pd.DataFrame(rows)

# TODO 2
within_capacity = pr[pr.alerts <= 150]
capacity_choice = within_capacity.sort_values(["recall", "precision"], ascending=False).iloc[0]
best_f1 = pr.loc[pr.f1.idxmax()]

# TODO 3
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(pr.recall, pr.precision, "-o", color="grey", ms=4)
for label, row, colour in [("current 0.50", pr[pr.threshold == 0.5].iloc[0], "tab:blue"),
                           (f"capacity-limited {capacity_choice.threshold:.2f}", capacity_choice, "tab:red"),
                           (f"best F1 {best_f1.threshold:.2f}", best_f1, "tab:green")]:
    ax.scatter(row.recall, row.precision, s=110, color=colour, zorder=3, label=label)
ax.set(xlabel="Recall", ylabel="Precision", title="Precision-recall curve over detector thresholds")
ax.legend()

plt.tight_layout(); plt.show()
display(pr.round(3))

# TODO 4
print(f"Recommendation: use threshold {capacity_choice.threshold:.2f} -> {int(capacity_choice.alerts)} alerts/month, "
      f"recall {capacity_choice.recall:.1%}, precision {capacity_choice.precision:.1%} "
      f"(best F1 would be {best_f1.threshold:.2f}, but it produces {int(best_f1.alerts)} alerts).")

**Reflection:** The capacity-constrained threshold and the best-F1 threshold may differ. Who should make that call — the
model or the Head of Fraud Ops — and why does visual analytics insist on keeping a human in the loop here (Slides 35–37)?

_Your answer:_

## Task 5 — An Interactive Tool for the Investigators (Request #5)

**Real-world usage:** Investigators triage flagged cases themselves with zoom, filter and hover-for-details.

**Why this technique:** **Details-on-demand / tooltips** (Slides 22–23), **dynamic queries** (Slide 15) and **drill-down**
(Slide 24) — Shneiderman's mantra as an actual tool.

**TODO:**
1. Interactive scatter (plotly) of all transactions with `detector_score >= 0.3`: x = timestamp, y = amount (log),
   colour = `label`, with a tooltip showing txn_id, customer, city, category, distance and score. Set `dragmode="select"`.
2. Interactive **sunburst** for drill-down: city → merchant category, sized by transaction count, coloured by fraud rate.

In [ ]:
cat_col = next(c for c in tx.columns if "categ" in c.lower())
cust_col = next((c for c in tx.columns if "cust" in c.lower()), None)
id_col = next((c for c in tx.columns if "txn" in c.lower() or c.lower().endswith("id")), tx.columns[0])

# TODO 1
flagged = tx[tx.detector_score >= 0.3]
hover_cols = [c for c in [id_col, cust_col, "city", cat_col, "distance_from_home_km", "detector_score"] if c]
fig1 = px.scatter(flagged, x="timestamp", y="amount_inr", color="label", log_y=True,
                  hover_data=hover_cols, color_discrete_map={"confirmed fraud": "crimson", "legitimate": "lightsteelblue"},
                  title=f"{len(flagged)} flagged transactions (score >= 0.3)")
fig1.update_layout(dragmode="select")
fig1.show()

# TODO 2
drill = (tx.groupby(["city", cat_col])
           .agg(transactions=("is_fraud", "size"), fraud_rate=("is_fraud", "mean"))
           .reset_index())
fig2 = px.sunburst(drill, path=["city", cat_col], values="transactions", color="fraud_rate",
                   color_continuous_scale="Reds", title="City -> merchant category (colour = fraud rate)")
fig2.show()

**Reflection:** Tooltips are great for one case at a time. Why would they be a *frustrating* way for an investigator to get
an overview of the 400 flagged cases (Slide 23), and which view from this notebook fixes that?

_Your answer:_

## Task 6 — Prove or Kill the "Fraud Ring" Hypothesis (Request #6)

**Real-world usage:** Before escalating to law enforcement, the team must rule out simpler explanations.

**Why this technique:** **Hypothesis testing** with visual + statistical confirmation and an explicit **counter-example
search** to guard against confirmation bias (Slides 42–43).

**TODO:**
1. State the hypothesis precisely using what you found in Task 1 (city and day range).
2. Compute the fraud rate inside vs. outside that city–day window and a chi-square p-value
   (`scipy.stats.chi2_contingency` on `pd.crosstab(in_window, tx.is_fraud)`).
3. Counter-example searches: fraud rate for (a) the same city on other days, (b) other cities on the same days.
4. Visual confirmation: a bar chart of the four groups (window / same city other days / other cities same days / rest).
5. Print your verdict.

In [ ]:
from scipy.stats import chi2_contingency

# TODO 1: hypothesis as a boolean mask
# Look at the Task 1 heatmap and edit these three values if your hot spot is different.
fraud_by_city_day = tx[tx.is_fraud == 1].groupby(["city", "day"]).size()
hot_city = fraud_by_city_day.groupby("city").sum().idxmax()
city_days = fraud_by_city_day.loc[hot_city]
hot_days = city_days[city_days >= 0.5 * city_days.max()].index
day_from, day_to = int(hot_days.min()), int(hot_days.max())
print(f"Hypothesis: a fraud ring in {hot_city} between day {day_from} and day {day_to}")

in_window = (tx.city == hot_city) & tx.day.between(day_from, day_to)

# TODO 2
table = pd.crosstab(in_window, tx.is_fraud)
chi2, p_value, _, _ = chi2_contingency(table)
rate_in, rate_out = tx.loc[in_window, "is_fraud"].mean(), tx.loc[~in_window, "is_fraud"].mean()
print(f"Fraud rate inside window {rate_in:.1%} vs outside {rate_out:.1%} (chi-square p = {p_value:.2e})")

# TODO 3
same_city_other_days = (tx.city == hot_city) & ~tx.day.between(day_from, day_to)
other_cities_same_days = (tx.city != hot_city) & tx.day.between(day_from, day_to)
rest = (tx.city != hot_city) & ~tx.day.between(day_from, day_to)
groups = {"window": in_window, "same city,\nother days": same_city_other_days,
          "other cities,\nsame days": other_cities_same_days, "rest": rest}
rates = {name: tx.loc[mask, "is_fraud"].mean() for name, mask in groups.items()}
print({k.replace("\n", " "): f"{v:.1%}" for k, v in rates.items()})

# TODO 4
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(rates.keys(), [v * 100 for v in rates.values()], color=["crimson", "grey", "grey", "lightgrey"])
ax.set(ylabel="Fraud rate (%)", title="Fraud rate: the window vs its counter-examples")

plt.tight_layout(); plt.show()

# TODO 5
holds = rates["window"] > 3 * max(rates["same city,\nother days"], rates["other cities,\nsame days"]) and p_value < 0.001
print("Verdict: hypothesis SUPPORTED - only that city AND those days are elevated." if holds
      else "Verdict: hypothesis NOT supported - fraud is not specific to that city-and-day window.")

**Reflection:** Which of your two counter-example searches would have *killed* the hypothesis if it had come out
differently, and what alternative explanation would it have supported instead?

_Your answer:_

## Task 7 — The Decision (No Code)

**Reflection (final):** Write a 4–6 sentence note to the Head of Fraud Ops covering the whole notebook. It should:
- State what happened last month in one sentence, with numbers.
- Recommend the alert threshold and justify it with capacity, precision/recall and the human-in-the-loop idea.
- Recommend the **one** interactive view to build first for the investigators, and justify it with at least one lecture
  idea (overview first, linked brushing, dynamic queries with the 0.1 s budget, details-on-demand).
- Name the feedback loop (Slide 51) that should be added so next month's detector is better.

_Your final note here:_